# 🐜 Ant Colony Simulation

**Authors:** José Pablo Castro-Delgado, Róisín Burke, Yu-Jou Yang, 

---

## 📋 Scenario Description

We simulate an ant colony facing a roaming predator. The colony consists of three roles: **foragers** that collect food pellets and bring energy back to the colony; **fighters** that patrol the colony perimeter and charge at the predator when it gets close; and a **queen** who sits stationary at the colony centre.

A single **predator** roams the map and hunts ants. All agents operate under an internal **energy** level that drains over time and gates behaviour intensity — a hungry predator hunts more aggressively, a stressed forager stops foraging and flees instead. Ants consumed by the predator are temporarily removed and respawn at the queen after a stun delay.

We expect to observe emergent colony-level patterns from these simple local rules: foragers oscillating between foraging and fleeing as the predator approaches; fighters clustering around the queen when an alarm is triggered; and predator energy driving rhythmic hunt-and-idle cycles.

---

## 🗺️ Entity Plan

| Entity type | Subtype label | Count | Color | Diameter | Role |
|------------|--------------|-------|-------|----------|---------|
| Agent | `"forager"` | 6 | goldenrod | 3 | Collect food, flee predator, return to queen |
| Agent | `"fighter"` | 4 | orangered | 4 | Patrol colony, attack predator on alarm |
| Agent | `"queen"` | 1 | mediumpurple | 7 | Stationary colony anchor, home reference |
| Agent | `"predator"` | 1 | crimson | 8 | Roaming threat, hunts foragers and fighters |
| Object | `"food"` | 24 | limegreen | 3 | Energy source, consumed by foragers, respawns |
| Object | `"obstacle"` | 8 | saddlebrown | 10 | Immovable blocks, navigation constraints |


---
## ⚙️ Section 1 — Setup

In [ ]:
from vivarium.controllers import VivariumController
controller = VivariumController.start_session(scene_name="miniproject")

INFO:vivarium.controllers.vivarium_controller:Server already running with scene 'miniproject'. Connecting.
INFO:vivarium.controllers.vivarium_controller:Controller thread started on client
INFO:vivarium.controllers.vivarium_controller:Controller thread is already running
INFO:vivarium.controllers.vivarium_controller:VivariumController session 'miniproject' is started


In [ ]:
# ── Defining subtype  ──────────────────────────────────
controller.set_subtype_labels([
    "forager",
    "fighter",
    "queen",
    "predator",
	"food",
	"obstacle",
])

# Verify
print("Available subtypes:", controller.subtypes)

Available subtypes: ['forager', 'fighter', 'queen', 'predator', 'food', 'obstacle', 'subtype_7', 'subtype_8']


In [ ]:
# ── Speed calibration ─────────────────────────────────────────────
# Increase if simulation feels too slow. Keep between 1 and 6.
controller.simulator.env.num_scan_steps = 2

---
## 🧩 Section 2 — Entity Initialisation

Assign subtypes, colors, sizes, and speeds to all 12 agents and 32 objects according to the entity plan above.

**Step 1 goal:** get all entities visible on the map with the right appearance before adding any behaviour. Agents `[0:6]` → foragers · `[6:10]` → fighters · `[10]` → queen (place at map centre, make immovable) · `[11]` → predator. Objects `[0:24]` → food · `[24:32]` → obstacles (make immovable).  
*(Agents = squares, Objects = circles on the map)*

In [ ]:
# ── Agent initialisation ──────────────────────────────────────────────────────
# 12 agents total: controller.agents[0:12]

# -- Agents: foragers --
for agent in controller.agents[0:6]:
    agent.subtype = 'forager'
    agent.diameter = 3
    agent.color = 'goldenrod'
    agent.max_speed = 2

# -- Agents: fighters --
for agent in controller.agents[6:10]:
    agent.subtype = 'fighter'
    agent.diameter = 4
    agent.color = 'orangered'
    agent.max_speed = 1.5

# -- Agents: queen --
queen = controller.agents[10]
queen.subtype = 'queen'
queen.diameter = 7
queen.color = 'purple'
queen.max_speed = 0

# -- Agents: predator --
predator = controller.agents[11]
predator.subtype = 'predator'
predator.diameter = 8
predator.max_speed = 1.2

print("Agents initialised.")

In [ ]:
# ── Object initialisation ─────────────────────────────────────────────────────
# 32 objects total: controller.objects[0:32]

# -- Resources (food) --
for obj in controller.objects[0:24]:
    obj.subtype = "food"
    obj.diameter = 3
    obj.color = "limegreen"

# -- Obstacles --
for obj in controller.objects[24:32]:
    obj.subtype = "obstacle"
    obj.diameter = 10
    obj.color = "saddlebrown"
    obj.mass = 1000       # make immovable
    obj.friction = 1000

print("Objects initialised.")

In [ ]:
# ── Verification ────────────────────────────────────────────────────────
print("=== Agents ===")
for agent in controller.agents:
    print(f"  subtype={agent.subtype}, diameter={agent.diameter}, color={agent.color}, max_speed={agent.max_speed}")

print("\n=== Objects (sample: first and last 4) ===")
for obj in list(controller.objects[0:4]) + list(controller.objects[28:32]):
    print(f"  subtype={obj.subtype}, diameter={obj.diameter}, color={obj.color}")

---
## 🤖 Section 3 — Behaviors

**Step 1 scope — foragers, food, obstacles (no predator yet):** implement `avoid_obstacles` and `forage` for foragers, and `wander` as a fallback when no food is in range. Verify foragers navigate around obstacles and collect food.

**Step 2 scope — add the predator:** implement `hunt_ants` for the predator (always hunting at this stage, no energy gating yet) and `flee_predator` for foragers. Add `patrol_colony` and `attack_predator` for fighters. Verify the predator chases ants and fighters respond.

Each behaviour function takes an `agent` and returns `(left_motor, right_motor)` in `[0, 1]`. Weights set here are the starting values; Section 4 routines will adjust them dynamically.

In [ ]:
# ── Behavior definitions ──────────────────────────────────────────────────────
# Each function takes an `agent` and returns (left_motor, right_motor) in [0, 1]

def avoid_obstacles(agent):
    left, right = agent.proximeters(sensed_entities=["obstacle"])
    return 1 - right, 1 - left          # shyness — veers away

def forage_food(agent):
    left, right = agent.proximeters(sensed_entities=["food"])
    return right, left                   # aggression — steers toward

def flee_predator(agent):
    left, right = agent.proximeters(sensed_entities=["predator"])
    return left, right                   # fear — direct excitatory (runs away)

def hunt_ants(agent):
    left, right = agent.proximeters(sensed_entities=["forager", "fighter"])
    return right, left                   # aggression — chases ants

def patrol_colony(agent):
    left, right = agent.proximeters(sensed_entities=["queen"])
    return 1 - right, 1 - left          # orbits away from queen = circles around it

def attack_predator(agent):
    left, right = agent.proximeters(sensed_entities=["predator"])
    return right, left                   # charges toward predator

print("Behaviors defined.")

In [ ]:
# ── Attach behaviors to agents ────────────────────────────────────────────────
# Always detach first to avoid duplicates when re-running this cell
for agent in controller.agents:
    agent.detach_all_behaviors(stop_motors=True)
    agent.detach_all_routines()

for agent in controller.agents:
    if agent.subtype == "forager":
        agent.attach_behavior(avoid_obstacles, weight=1.0)
        agent.attach_behavior(forage_food,     weight=1.0)
        agent.attach_behavior(flee_predator,   weight=1.5)

    elif agent.subtype == "fighter":
        agent.attach_behavior(avoid_obstacles, weight=1.0)
        agent.attach_behavior(patrol_colony,   weight=0.8)
        agent.attach_behavior(attack_predator, weight=1.5)

    elif agent.subtype == "predator":
        agent.attach_behavior(avoid_obstacles, weight=1.0)
        agent.attach_behavior(hunt_ants,       weight=1.0)

print("Behaviors attached.")

---
## 🔄 Section 4 — Internal States & Routines

**Step 3 scope — energy-weighted behaviours:** introduce `energy` as the primary drive modulator for all agents, and `stress` for foragers only.

- **Energy** drains each tick at a fixed rate per agent type. Foragers and fighters recover energy by consuming food; the predator recovers by consuming ants. At low energy, foraging/hunting weights increase; at high energy, idle/wander weights dominate.
- **Stress** (foragers only) accumulates when the predator is nearby and decays passively. Above the stress threshold, the `forage` weight drops and `flee_predator` weight spikes.
- Use `energy_diameter` as a visual health indicator — agents shrink as they lose energy.

Remember: initialise all `agent.internal` fields *before* attaching routines.

In [ ]:
# ── Routine definitions ───────────────────────────────────────────────────────

# Energy routine: decays over time, recovers when consuming a resource
def energy_routine(agent):
    agent.internal.energy -= 0.0002                          # slow decay
    consumed = agent.has_consumed()                          # how many consumed this step
    agent.internal.energy += 0.1 * consumed                 # recover on consumption
    agent.internal.energy = max(0.0, min(1.0, agent.internal.energy))  # clamp [0,1]

# Visual feedback: map energy → diameter
def energy_diameter(agent):
    agent.diameter = 1 + agent.internal.energy * 4          # diameter 1–5

# Dynamic behavior weight: hungry agents forage harder
def dynamic_foraging_weight(agent):
    w = 2.0 - agent.internal.energy * 1.5                   # weight 0.5–2.0
    agent.change_behavior_weight(forage_food, new_weight=w)

# ── Add your own routines below ───────────────────────────────────────────────
# def my_routine(agent):
#     ...

print("Routines defined.")


In [ ]:

# Internal state init
for agent in controller.agents:
    if agent.subtype in ("forager", "fighter"):
        agent.internal.energy = 0.8
        agent.internal.stress = 0.0   # foragers only, but harmless on fighters

    elif agent.subtype == "predator":
        agent.internal.energy = 0.5

In [ ]:
# ── Attach routines to agents ─────────────────────────────────────────────────
for agent in controller.agents:
    if agent.subtype in ("forager", "fighter"):
        agent.attach_routine(energy_routine)
        agent.attach_routine(energy_diameter)
        agent.attach_routine(dynamic_foraging_weight)

    elif agent.subtype == "predator":
        agent.attach_routine(energy_routine)
        agent.attach_routine(energy_diameter)

print("Routines attached.")

---
## 🌱 Section 5 — Environmental Dynamics

Configure the consumption and spawning mechanisms that keep the simulation alive over time.

- **Slot 1 consumption:** foragers consume food pellets on contact — this is what triggers energy recovery in the forager energy routine.
- **Slot 2 consumption:** the predator consumes foragers — this removes the ant from the map (exists = False), which the stun/respawn controller routine in Section 6 will detect and handle.
- **Slot 3 consumption:** the predator consumes fighters — same stun logic applies.
- **Slot 1 spawn:** food pellets respawn periodically so the colony always has something to forage for. Tune the period to control how abundant food is.

In [ ]:
# ── Consumption ─────────────────────────────────────────────────────────
# Slot 1: foragers eat food
controller.consumption.slot_1.source_subtype = "forager"
controller.consumption.slot_1.target_subtype = "food"
controller.consumption.slot_1.range = 1
controller.consumption.slot_1.start = True

# Slot 2: predator eats foragers
controller.consumption.slot_2.source_subtype = "predator"
controller.consumption.slot_2.target_subtype = "forager"
controller.consumption.slot_2.range = 1
controller.consumption.slot_2.start = True

# Slot 3: predator eats fighters
controller.consumption.slot_3.source_subtype = "predator"
controller.consumption.slot_3.target_subtype = "fighter"
controller.consumption.slot_3.range = 1
controller.consumption.slot_3.start = True

In [ ]:
# ── Spawning slots ────────────────────────────────────────────────────────────

# Food respawns every 100 steps
controller.spawn.slot_1.subtype = "food"
controller.spawn.slot_1.period = 100
controller.spawn.slot_1.start = True

---
## 🔁 Section 6 — Controller Routines (Simulation-Wide Logic)

**Step 4 scope — colony complexity:** layer in colony-level emergent behaviour using controller routines that coordinate across multiple agents.

- **Stun & respawn:** when the predator consumes an ant, Vivarium sets `agent.exists = False`. This routine detects those ants, counts down a stun timer, then teleports them back to the queen with partial energy restored — simulating a stunned ant recovering at the colony.
- **Territorial alarm:** approximates the alarm pheromone from the project plan. If the predator enters a radius around the queen, all fighters immediately receive maximum attack weight — overriding the weight routine — regardless of how close they personally are to the predator.
- **Population logging:** records how many foragers, fighters, and food pellets exist each tick, plus predator energy, so we can plot colony dynamics afterwards.

In [ ]:
# ── Controller routine: population logging ─────────────────────────────────
def log_populations(controller):
    n_foragers = len([a for a in controller.agents if a.subtype == "forager" and a.exists])
    n_fighters = len([a for a in controller.agents if a.subtype == "fighter" and a.exists])
    n_food     = len([o for o in controller.objects if o.subtype == "food" and o.exists])
    controller.logger.add("forager_count", n_foragers)
    controller.logger.add("fighter_count", n_fighters)
    controller.logger.add("food_count",    n_food)



# ── Controller routine: Stun and Respawn ───────────────────────────
STUN_DURATION = 150   # ticks before respawn

def stun_and_respawn(controller):
    queen = next(a for a in controller.agents if a.subtype == "queen")
    for agent in controller.agents:
        if agent.subtype in ("forager", "fighter") and not agent.exists:
            if not hasattr(agent.internal, "stun_timer"):
                agent.internal.stun_timer = STUN_DURATION
            agent.internal.stun_timer -= 1
            if agent.internal.stun_timer <= 0:
                agent.exists       = True
                agent.x_position   = queen.x_position
                agent.y_position   = queen.y_position
                agent.internal.energy     = 0.4
                agent.internal.stun_timer = STUN_DURATION


In [ ]:
# ── Attach controller routines ────────────────────────────────────────────────
controller.attach_routine(stun_and_respawn)
controller.attach_routine(log_populations, interval=10)

print("Controller routines attached.")
controller.print_routines()

---
## 📊 Section 7 — Data Logging & Plots

Visualize what is happening in the simulation using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# ── Plot population dynamics ──────────────────────────────────────────────────
plt.plot(controller.logger.get("forager_count"), label="foragers", color="goldenrod")
plt.plot(controller.logger.get("fighter_count"), label="fighters", color="orangered")
plt.plot(controller.logger.get("food_count"),    label="food",     color="limegreen")
plt.xlabel("Time (×10 steps)")
plt.ylabel("Count")
plt.title("Colony population dynamics")
plt.legend()
plt.tight_layout()
plt.show()

---
## 🧹 Section 8 — Utilities

Handy one-off cells to inspect or reset the simulation state.

In [ ]:
# ── Inspect current state of all agents ───────────────────────────────────────
for agent in controller.agents:
    exists_str = "✓" if agent.exists else "✗"
    energy_str = f", energy={agent.internal.energy:.2f}" if hasattr(agent.internal, 'energy') else ""
    print(f"[{exists_str}] {agent.subtype:10s} | d={agent.diameter} | speed={agent.max_speed}{energy_str}")

In [ ]:
# ── Stop everything (use when experimenting) ──────────────────────────────────
for agent in controller.agents:
    agent.detach_all_behaviors(stop_motors=True)
    agent.detach_all_routines()

controller.consumption.slot_1.start = False
controller.consumption.slot_2.start = False
controller.consumption.slot_3.start = False  
controller.spawn.slot_1.start = False
controller.spawn.slot_2.start = False


# Detach controller routines too
controller.detach_all_routines()

print("All behaviors, routines and mechanisms stopped.")

In [ ]:
# ── Clear all logged data ──────────────────────────────────────────────────────
for topic in ["forager_count", "fighter_count", "food_count"]:
    controller.logger.clear(topic)

for agent in controller.agents:
    for topic in ["energy", "x", "y"]:
        agent.logger.clear(topic)

print("Logs cleared.")

---
## 📝 Section 9 — Conclusion

> _Write your conclusions here after running the simulation._

**What did you observe?**  
> ...

**Did the simulation match your initial expectations?**  
> ...

**What would you have liked to add or improve?**  
> ...

**Limitations or issues encountered:**  
> ...

---

## 📚 Quick Reference

| Task | Code |
|------|------|
| Attach behavior | `agent.attach_behavior(fn, weight=1.0)` |
| Detach all behaviors | `agent.detach_all_behaviors(stop_motors=True)` |
| Attach routine | `agent.attach_routine(fn, interval=1)` |
| Detach all routines | `agent.detach_all_routines()` |
| Sense entities | `agent.proximeters(sensed_entities=["subtype"])` |
| Change behavior weight | `agent.change_behavior_weight(fn, new_weight=2.0)` |
| Check consumed | `agent.has_consumed()` |
| Log data | `agent.logger.add("topic", value)` |
| Retrieve data | `agent.logger.get("topic")` |
| Clear log | `agent.logger.clear("topic")` |
| Toggle existence | `agent.exists = True / False` |
| Set position | `agent.x_position = 50; agent.y_position = 50` |
| Consumption slot | `controller.consumption.slot_1.start = True/False` |
| Spawn slot | `controller.spawn.slot_1.start = True/False` |
| Controller routine | `controller.attach_routine(fn, interval=10)` |
